## Base Import and API modules

In [ ]:
import requests
import json
import random
from faker import Faker

from contextlib import redirect_stdout
import io

# Global variables for pagination, easy to tweak here
DEFAULT_PAGE = 0
DEFAULT_SIZE = 25
BASE_URL = "http://localhost:8080/api"

general_game = "Undertale"
fake_friends = 10
fake_requests = 5

invalid_id = "AAAABBBBCCCCDDDDEEEEFFFF"

puppet_users = []
popular_games = []

def print_response(response):
    """Utility to print responses clearly (indented JSON or plain string)"""
    print(f"Status Code: {response.status_code}")
    try:
        # Try to parse and print nicely formatted JSON
        parsed_json = response.json()
        print(json.dumps(parsed_json, indent=4))
    except ValueError:
        # If it fails (e.g., plain text like "The game has been added"), print raw text
        text = response.text
        if text:
            print(text)
        else:
            print("[No content in response body]")
    print("=" * 50 + "\n")

def print_json(target_json):
    print(json.dumps(target_json, indent=4))

def mute(func, *args, **kwargs):
    """Prevents output of function func"""
    with redirect_stdout(io.StringIO()):
        return func(*args, **kwargs)

def login_as(username, output = True):
    api.logout(False)
    my_payload = {
        "email" : username + "@hotmail.com",
        "password" : username
    }
    response = api.login(my_payload,output)
    if(response.status_code != 200):
        raise ValueError("ERROR DURING LOGIN")
    else:
        if(output):
            print("You are now logged as: " + username)
            print()

def admin_login(output = True):
    api.logout(False)
    my_payload = {
        "email" : "admin@hotmail.com",
        "password" : "admin"
    }
    response = api.login(my_payload,output)
    if(response.status_code != 200):
        raise ValueError("ERROR DURING LOGIN")
    else:
        if(output):
            print("You are now logged as Admin")
            print()
    

def get_user_id(username):
    src_res = api.search_user(username,0,1).json()["searchResult"]
    if(len(src_res) != 1):
        raise ValueError("USER NOT FOUND")
    return src_res[0]["id"]

def get_game_id(game_name):
    src_res = api.search_game(game_name,0,1).json()["searchResult"]
    if(len(src_res) != 1):
        raise ValueError("GAME NOT FOUND")
    return src_res[0]["id"]

def get_user_game_review(user,game_id, can_fail = True):
    review_id = ""

    response = api.get_game_reviews(game_id).json()["reviews"]

    for r in response:
        if(r["username"] == user):
            review_id = r["id"]
            break

    if(review_id == "" and can_fail):
        raise ValueError("NO MATCHING REVIEW WAS FOUND")
    
    return review_id

def create_unique_user():
    api.logout(False)
    fake = Faker()
    for i in range(0,5):
        while(True):
            register_username = fake.user_name() + f"{random.randint(1000, 9999)}" # avoid duplicates
            user_payload = {
                "username" : register_username,
                "email" : register_username + "@hotmail.com",
                "password" : register_username,
                "birthDate" : "2002-08-29"
            }

            response = api.register(user_payload)
            if(response.status_code == 200):
                break
        return register_username

def create_fake_friends(username,friends_number, requests_number = 0):

    neo4j_string = "OPTIONAL MATCH (anchor:User {username : \"" + username + "\"}) \n"
    relationship_string = "OPTIONAL MATCH p = (anchor)-[r:FRIENDS_WITH]-(friend:User) \nRETURN anchor, p"

    anchor = username
    anchor_id = get_user_id(username)

    user_list = []
    user_list.append(anchor)

    user_ids = []

    for i in range(0,friends_number + requests_number):
        
        u = create_unique_user()

        login_as(u, False)

        api.send_friend_request(anchor_id)
        if(i < friends_number):
            user_list.append(u)
            user_ids.append(get_user_id(u))

        neo4j_string += f"MATCH (u{i}:User" + " {username : \"" + u + "\"}) \n"
        relationship_string += f", u{i}"

    # accepting friend requests
    login_as(anchor, False)

    for i in range(0, friends_number):
        api.approve_friend_request(user_ids[i])

    if(api.get_own_profile().json()["friends"] != friends_number):
        raise ValueError("WRONG FRIEND VALUE")
    else:
        print()
        print(f">>>>> {friends_number} FRIENDSHIPS ESTABLISHED!")
        if(requests_number > 0):
            print(f">>>>> {requests_number} REQUESTS CREATED!")
        

    queries = {
        "mongo" : build_mongo_query(user_list),
        "neo4j" : neo4j_string + relationship_string
    }   
    
    return queries

def add_games_batch(username,game_list):
    
    login_as(username,False)

    i = 1

    for game in game_list:
        game_id = get_game_id(game)
        payload = {
            "gameId" : game_id,
            "achievements" : 0,
            "hoursPlayed" : i*10
        }
        i+=1
        resp = api.edit_library(payload)
        if(resp.status_code != 200):
            raise ValueError("ERROR!")
        
        review = {
            "reviewText" : "fortnite and " + game,
            "score" : 5
        }
        resp = api.add_review(game_id,review)
        if(resp.status_code != 200):
            raise ValueError("ERROR!")

    
    api.logout(False)

def print_friend_query(username):
    print("\nNEO4J QUERY:\n")
    print("MATCH (anchor:User {username : \"" + username + "\"}) \n" +
        "OPTIONAL MATCH p = (anchor)-[r:FRIENDS_WITH]-(friend:User) \n"+ 
        "RETURN anchor, p\n" )
    
def print_neo_query(username):
    print("\nNEO4J QUERY:\n")
    print("MATCH (anchor:User {username : \"" + username + "\"}) \n" +
        "OPTIONAL MATCH p = (anchor)-[]-() \n"+ 
        "RETURN anchor, p\n" )
    
def build_mongo_query(user_list):

    query = "{ username: { $in: [ \""  + "\" , \"".join(user_list) + "\" ]} }"

    return query
    
def print_mongo_query(user_list):

    query = build_mongo_query(user_list)

    print("\nMONGO QUERY:\n")
    print(query + "\n")


class PlayerHiveClient:
    def __init__(self, base_url):
        self.base_url = base_url
        self.session = requests.Session()
        self.session.headers.update({"Content-Type": "application/json"})
        self.token = None

    def set_token(self, token, output):
        """Sets the JWT token for authenticated requests"""
        self.token = token
        self.session.headers.update({"Authorization": f"Bearer {token}"})
        if(output):
            print("[+] Token set successfully. You are now authenticated.\n")

    def logout(self, output = True):
        """Clears the token to allow switching accounts or testing unauthenticated routes"""
        self.token = None
        if "Authorization" in self.session.headers:
            del self.session.headers["Authorization"]
        if(output):
            print("[-] Token cleared. You are now logged out.\n")

    def _request(self, method, endpoint, **kwargs):
        """Base method to execute and print the request"""
        url = f"{self.base_url}{endpoint}"
        #print(f"--- {method} {url} ---") # uncomment to see the url
        response = self.session.request(method, url, **kwargs)
        return response

#### API MODULES
class API(PlayerHiveClient):
    
    # ================= AUTH =================
    def register(self, payload={}):
        return self._request("POST", "/auth/register", json=payload)

    def login(self, payload={}, output = True):
        response = self._request("POST", "/auth/login", json=payload)
        if response.status_code == 200:
            token = response.json().get("accessToken")
            if token:
                self.set_token(token, output)
        return response

    # ================= USER =================
    def get_own_profile(self):
        return self._request("GET", "/user/MyProfile")

    def get_user_profile(self, user_id):
        return self._request("GET", f"/user/{user_id}")
    
    def get_own_library(self,page=DEFAULT_PAGE, size=DEFAULT_SIZE):
        params = {"page": page, "size": size}
        return self._request("GET", f"/user/MyLibrary", params=params)

    def get_user_library(self, user_id, page=DEFAULT_PAGE, size=DEFAULT_SIZE):
        params = {"page": page, "size": size}
        return self._request("GET", f"/user/library/{user_id}", params=params)

    def edit_library(self, payload={}):
        return self._request("POST", "/user/editLibrary", json=payload)

    def remove_from_library(self, game_id):
        return self._request("DELETE", f"/user/removeFromLibrary/{game_id}")

    def get_friend_list(self, user_id, page=DEFAULT_PAGE, size=DEFAULT_SIZE):
        params = {"page": page, "size": size}
        return self._request("GET", f"/user/friends/{user_id}", params=params)
    
    def get_own_friend_list(self,page=DEFAULT_PAGE, size=DEFAULT_SIZE):
        params = {"page": page, "size": size}
        return self._request("GET", f"/user/MyFriends", params=params)
    

    def get_friend_requests(self,page=DEFAULT_PAGE,size=DEFAULT_SIZE):
        params = {"page": page, "size": size}
        return self._request("GET", "/user/friendRequests", params = params)

    def search_user(self, query, page=DEFAULT_PAGE, size=DEFAULT_SIZE):
        params = {"page": page, "size": size}
        return self._request("GET", f"/user/search/{query}", params=params)

    def send_friend_request(self, target_user_id):
        return self._request("POST", f"/user/sendFriendRequest/{target_user_id}")

    def approve_friend_request(self, target_user_id):
        return self._request("POST", f"/user/approveFriendRequest/{target_user_id}")

    def deny_friend_request(self, target_user_id):
        return self._request("DELETE", f"/user/denyFriendRequest/{target_user_id}")

    def remove_friend(self, friend_id):
        return self._request("DELETE", f"/user/removeFriend/{friend_id}")
    
    def get_user_reviews(self, user_id, page=DEFAULT_PAGE,size=DEFAULT_SIZE):
        params = {"page": page, "size": size}
        return self._request("GET","/user/reviews/" + user_id, params = params)
    
    def get_own_reviews(self, page=DEFAULT_PAGE,size=DEFAULT_SIZE):
        params = {"page": page, "size": size}
        return self._request("GET","/user/MyReviews", params = params)

    def delete_account(self):
        return self._request("DELETE", "/user/deleteAccount")

    # ================= GAMES =================
    def get_game_info(self, game_id):
        return self._request("GET", f"/games/{game_id}")

    def search_game(self, game_name, page=DEFAULT_PAGE, size=DEFAULT_SIZE):
        params = {"page": page, "size": size}
        return self._request("GET", f"/games/search/{game_name}", params=params)

    def get_game_reviews(self, game_id, page=DEFAULT_PAGE, size=DEFAULT_SIZE):
        params = {"page": page, "size": size}
        return self._request("GET", f"/games/reviews/{game_id}", params=params)

    def add_review(self, game_id, payload={}):
        return self._request("POST", f"/games/addReview/{game_id}", json=payload)

    def delete_review(self, review_id):
        return self._request("DELETE", f"/games/deleteReview/{review_id}")

        # INTERESTING QUERIES
    def get_game_recommendations(self):
        return self._request("GET","/games/getRecommendations")
    
    def get_trending_games(self):
        return self._request("GET","/admin/getTrending")

    # ================= ADMIN =================
    def add_game(self, payload={}):
        return self._request("POST", "/admin/addGame", json=payload)

    def edit_game(self, game_id, payload={}):
        return self._request("PATCH", f"/admin/editGame/{game_id}", json=payload)

    def delete_game(self, game_id):
        return self._request("DELETE", f"/admin/deleteGame/{game_id}")

    def delete_user_admin(self, user_id):
        return self._request("DELETE", f"/admin/deleteUser/{user_id}")

# Initialize the API object
api = API(BASE_URL)

general_user = create_unique_user()
general_user_2 = create_unique_user()

print("user1: " + general_user) 
print("user2: "+ general_user_2)
print_neo_query(general_user)
print_neo_query(general_user_2)

print_mongo_query([general_user,general_user_2])

# a = create_fake_friends(general_user,fake_friends,fake_requests)
# a = create_fake_friends(general_user_2,fake_friends,fake_requests)


# add_games_batch(general_user,["Undertale", "Fantasy General II","Tank Rampage"])
# add_games_batch(general_user_2,["Ramen Oil Pecking Simulator", "Thunderbolt 2","Funguys Swarm"])


## LOGIN SECTION

### Register user

In [ ]:
api.logout(False)

print(">>> REGISTER AS STANDARD USER")
fake = Faker()

register_username = fake.user_name() # avoid duplicates
user_payload = {
    "username" : register_username,
    "email" : register_username + "@hotmail.com",
    "password" : register_username,
    "birthDate" : "2002-08-29"
}
print(">>>>> Registering as: "+register_username)
print_response(api.register(user_payload))

print(">>> SAME USERNAME TEST")

user_payload = {
    "username" : register_username,
    "email" : "eyes_emoji" + "@hotmail.com",
    "password" : register_username,
    "birthDate" : "2002-08-29"
}

print_response(api.register(user_payload))

print(">>> SAME EMAIL TEST")

user_payload = {
    "username" : "eyes_emoji",
    "email" : register_username + "@hotmail.com",
    "password" : register_username,
    "birthDate" : "2002-08-29"
}

print_response(api.register(user_payload))

print(">>> FRESH USER DELETION")
login_as(register_username)
print_response(api.delete_account())


### General Login

In [ ]:
api.logout()

print(">>> LOGIN AS STANDARD USER")
username = "fabriziomaggi"
user_payload = {
    "email" : username + "@hotmail.com",
    "password" : username

}
print_response(api.login(user_payload))

## USER CONTROLLER TESTS

### User Search Test

In [ ]:
print(">>> USER SEARCH")

api.logout(False)

page_number = 0
page_size = 10

user_search_target_string = "william"
user_search_result = api.search_user(user_search_target_string,page_number,page_size)

print_response(user_search_result)
# print_json(user_search_result.json()["content"])

print(">>> EMPTY USER SEARCH")

user_search_target_string = "&&&&&"
user_search_result = api.search_user(user_search_target_string,page_number,page_size)

print_response(user_search_result)

### Pagination Test (Input validation)

In [ ]:
print(">>> PAGINATION VALUES TEST\n")

print(">>> NEGATIVE PAGE NUMBER")

api.logout(False)

page_number = -1
page_size = 10

user_search_target_string = "br"
user_search_result = api.search_user(user_search_target_string,page_number,page_size)

print_response(user_search_result)

print(">>> PAGE SIZE 0")

page_number = 0
page_size = 0

user_search_result = api.search_user(user_search_target_string,page_number,page_size)

print_response(user_search_result)

print(">>> NEGATIVE PAGE SIZE")

page_number = 0
page_size = -1

user_search_result = api.search_user(user_search_target_string,page_number,page_size)

print_response(user_search_result)

print(">>> OUT OF BOUNDS PAGE NUMBER VALUE")

page_number = 100000
page_size = 10

user_search_result = api.search_user(user_search_target_string,page_number,page_size)

print_response(user_search_result)

print(">>> OUT OF BOUNDS PAGE SIZE VALUE")

page_number = 0
page_size = 10000

user_search_result = api.search_user(user_search_target_string,page_number,page_size)

print_response(user_search_result)

print(">>> MULTIPLE WRONG VALUES")

page_number = -1
page_size = -1

user_search_result = api.search_user(user_search_target_string,page_number,page_size)

print_response(user_search_result)

### Profile Request

In [ ]:
print(">>> PROFILE REQUEST")

api.logout(False)

username = "gturner"
#username = general_user

user_id = get_user_id(username)

print_response(api.get_user_profile(user_id))

### Non Existent Profile Request

In [ ]:
print(">>> NON EXISTENT PROFILE REQUEST")

api.logout(False)

print_response(api.get_user_profile(invalid_id))

### Own Profile Request

In [ ]:
print(">>> OWN PROFILE REQUEST, BOTH WITH ID AND WITHOUT")

#username = "gturner"
username = general_user

login_as(username)

user_id = get_user_id(username)

my_profile_without_id = api.get_user_profile(user_id)
my_profile_with_id = api.get_own_profile()

print_response(my_profile_with_id)
if(my_profile_without_id.json() == my_profile_with_id.json()):
    print("=========== The two queries are equal ==========")
else:
    print("======= ERROR! WITHOUT ID:")
    print_response(my_profile_without_id)

### Authentication Test: "My" Queries

In [ ]:
print(">>> \"MY\" QUERIES WITH NO ACCOUNT")

api.logout()

print_response(api.get_own_profile())
print_response(api.get_own_library(0,10))
print_response(api.get_own_reviews(0,10))

### Library Request

In [ ]:
print(">>> LIBRARY REQUEST")

api.logout(False)

#username = "gturner"
username = general_user

target_user = get_user_id(username)

page_number = 0
page_size = 25

print_response(api.get_user_library(target_user, page_number, page_size))

### My Library Request

In [ ]:
print(">>> OWN LIBRARY REQUEST")

#username = ""
username = general_user

login_as(username)

page_number = 0
page_size = 10

print_response(api.get_own_library( page_number, page_size))

### Non Existent Library Request

In [ ]:
print(">>> NON EXISTENT LIBRARY REQUEST")

api.logout(False)

page_number = 0
page_size = 2

print_response(api.get_user_library(invalid_id, page_number, page_size))

print(">>> INVALID ID LIBRARY REQUEST")

print_response(api.get_user_library("bla", page_number, page_size))


### Library Manipulation

In [ ]:
print(">>> ADDING GAME TO LIBRARY")

api.logout(False)
username = create_unique_user()
#username = ""

game_name = general_game
#game_name = "ULTRAKILL"
achievements = 0
hours = 10
hours2 = 100

# ADD QUERY PRINT

login_as(username,False)

print("\nNEO4J QUERY:\n")
print("MATCH (u:User {username : \"" + username + "\"}) \n"+
      "MATCH (g:Game {name : \"" + game_name + "\"}) \n"
    "OPTIONAL MATCH p=(u)-[r:PLAYED]->(game:Game) RETURN u,g,p\n\n")

game_id = get_game_id(game_name)

payload = {
    "gameId" : game_id,
    "achievements" : achievements,
    "hoursPlayed" : hours
}

print_response(api.edit_library(payload))


print(">>> EDIT GAME IN LIBRARY")

payload["hoursPlayed"] = hours2

print_response(api.edit_library(payload))

print(">>> REMOVE GAME FROM LIBRARY")

print_response(api.remove_from_library(game_id))


### Library Corner Cases Test

In [ ]:
print(">>> NO ACCOUNT LIBRARY MANIPULATION")

api.logout(False)

payload ={
    "gameId":invalid_id,
    "achievements":"0",
    "hoursPlayed":"10"
}

print_response(api.edit_library(payload))


print(">>> GHOST GAME LIBRARY MANIPULATION")

login_as(general_user,False)

print_response(api.edit_library(payload))


print(">>> GHOST GAME REMOVAL")

print_response(api.remove_from_library(invalid_id))


print(">>> BAD ACHIEVEMENT VALUE")

payload["gameId"] = get_game_id(general_game)
payload["achievements"] = 9999
print_response(api.edit_library(payload))

payload["achievements"] = -1
print_response(api.edit_library(payload))

### Friend List Test

In [ ]:
print(">>> PAGED FRIEND LIST")

username = general_user

page_number = 0
page_size = 2

api.logout(False)

user_id = get_user_id(username)

print_response(api.get_friend_list(user_id,page_number,page_size))

print(">>> OWN PAGED FRIEND LIST")

login_as(general_user)

print_response(api.get_own_friend_list(page_number,page_size))

print(">>> NO LOGIN MY FRIEND LIST")

api.logout()

print_response(api.get_own_friend_list(page_number,page_size))

print(">>> NO USER FRIEND LIST")

print_response(api.get_friend_list(invalid_id,page_number,page_size))


### Friend Requests Test

In [ ]:
print(">>> PAGED FRIEND REQUESTS")

#username = "simpsoneileen"
username = general_user

login_as(username)

page_number = 0
page_size = 5

print_response(api.get_friend_requests(page_number,page_size))

print(">>> NO LOGIN FRIEND REQUESTS")

api.logout()

print_response(api.get_friend_requests(page_number,page_size))


### Accepting Deleted User Friend Request

In [ ]:
print(">>> GHOST REQUEST ACCEPTANCE")

api.logout(False)

user1 = create_unique_user()
user1_id = get_user_id(user1)

user2 = create_unique_user()
user2_id = get_user_id(user2)

query_string = "{ username: { $in: [\"" + user1 +"\", \"" + user2 + "\"]} }"

print("MONGODB QUERY:\n")
print(query_string + "\n\n")

login_as(user1, False)
api.send_friend_request(user2_id)

api.delete_account()

login_as(user2, False)
print(">>> ACCEPTING FIRST TIME (GHOST REQUEST IS PRESENT)")
print_response(api.approve_friend_request(user1_id))

print(">>> ACCEPTING SECOND TIME (GHOST REQUEST IS NOT PRESENT)")
print_response(api.approve_friend_request(user1_id))

### Friend Requests Array Pagination

In [ ]:
print(">>> ARRAY PAGINATION TEST")

username = general_user

login_as(username)

print(">>> NEGATIVE PAGE NUMBER")

page_number = -1
page_size = 2

print_response(api.get_friend_requests(page_number,page_size))

print(">>> SIZE OUT OF BOUNDS")

page_number = 0
page_size = 10000

print_response(api.get_friend_requests(page_number,page_size))

print(">>> SIZE ZERO")

page_number = 0
page_size = 0

print_response(api.get_friend_requests(page_number,page_size))

print(">>> NEGATIVE SIZE")

page_number = 0
page_size = -1

print_response(api.get_friend_requests(page_number,page_size))

print(">>> OUT OF BOUNDS TEST")

page_number = 100
page_size = 10

print_response(api.get_friend_requests(page_number,page_size))


### Friend Manipulation

In [ ]:
print(">>> SEVERAL FRIEND MANAGEMENT TESTS")

user1 = create_unique_user()
user2 = create_unique_user()

api.logout(False)

user1_id = get_user_id(user1)
user2_id = get_user_id(user2)

if(user1_id == user2_id):
    raise ValueError("THE USERS ARE THE SAME!")

print_mongo_query([user1,user2])

print(">>>> NEO4J QUERY:\n\nMATCH (n1:User {username:\""+user1+"\"}) \n"+
      "MATCH (n2:User {username:\""+user2+"\"}) \n"+
      "OPTIONAL MATCH p=(n1)-[r:FRIENDS_WITH]-(n2) \n"+
      "RETURN n1, n2, p")


print()
print(">>> SEND FRIEND REQUEST")

login_as(user1)

print_response(api.send_friend_request(user2_id))

print(">>> DENY FRIEND REQUEST")

login_as(user2)

print_response(api.deny_friend_request(user1_id))

print(">>> ACCEPT FRIEND REQUEST")

login_as(user1)

api.send_friend_request(user2_id)

login_as(user2)

print_response(api.approve_friend_request(user1_id))

print(">>> REMOVE FRIEND")

login_as(user1)

print_response(api.remove_friend(user2_id))

print(">>> TWO-WAY FRIEND REQUEST")

login_as(user1)

print_response(api.send_friend_request(user2_id))

login_as(user2)

print_response(api.send_friend_request(user1_id))

print_response(api.remove_friend(user1_id))

print(">>> DOUBLE FRIEND REQUEST")

print_response(api.send_friend_request(user1_id))
print_response(api.send_friend_request(user1_id))

login_as(user1, False)
api.deny_friend_request(user2_id)


### Friend Management Corner Cases

In [ ]:
print(">>> NO LOGIN FRIEND MANAGEMENT")

api.logout()

user_id = get_user_id(general_user)

print(">>>>> SEND")
print_response(api.send_friend_request(user_id))

print(">>>>> ACCEPT")
print_response(api.approve_friend_request(user_id))

print(">>>>> DENY")
print_response(api.deny_friend_request(user_id))

print(">>>>> REMOVE FRIEND")
print_response(api.remove_friend(user_id))

print(">>> SELF FRIEND REMOVAL")
login_as(general_user)

print_response(api.remove_friend(user_id))

print(">>> SELF FRIEND REQUEST")

print_response(api.send_friend_request(user_id))

print(">>> SEND FRIEND REQUEST TO NO ONE")

print_response(api.send_friend_request(invalid_id))

print(">>> ACCEPT GHOST FRIEND REQUEST")
print_response(api.approve_friend_request(invalid_id))

### User Reviews

In [ ]:
print(">>> PAGED USER REVIEWS")

api.logout()

#username = ""
username = general_user
user_id = get_user_id(username)

page_number =0
page_size = 8

print_response(api.get_user_reviews(user_id,page_number,page_size))

print(">>> OWN REVIEWS")

login_as(username)

print_response(api.get_own_reviews(page_number,page_size))

## GAME CONTROLLER TEST

### Game Request

In [ ]:
print(">>> GAME REQUEST BY ID")

api.logout()

game_id = get_game_id(general_game)

response1 = api.get_game_info(game_id)

print_response(response1)

print(">>> LOGIN GAME TEST")

login_as(general_user)

response2 = api.get_game_info(game_id)

if(response1.json() != response2.json()):
    raise ValueError("WRONG RESPONSE")
else:
    print()
    print(">>>>>>> CORRECT RESPONSE")
    print()

### Non Existent Game Test

In [ ]:
api.logout()

print(">>> GHOST GAME TEST")

print_response(api.get_game_info(invalid_id))

print(">>> INPUT VALIDATION")

print_response(api.get_game_info("Hola"))

### Game Search

In [ ]:
api.logout()

print(">>> SEARCH GAME TEST")

game_target_string = "Under"
page_number = 0
page_size = 3

print_response(api.search_game(game_target_string, page_number, page_size))

print(">>> EMPTY GAME SEARCH")

print_response(api.search_game("&&&&", page_number, page_size))


### Game Reviews

In [ ]:
api.logout()

print(">>> GAME REVIEWS REQUEST TEST")

game = general_game
#game = "ULTRAKILL"
game_id = get_game_id(game)

page_number = 0
page_size = 25

print_response(api.get_game_reviews(game_id,page_number,page_size))

### Review Manipulation

In [ ]:
api.logout()

game = general_game
user = create_unique_user()

game_id = get_game_id(game)
login_as(user)

review = {
    "reviewText" : "i am " + user + ", i like apples, tangerines and "+ game,
    "score" : 9.5
}


print(">>> ADD GAME REVIEW TEST")

print_response(api.add_review(game_id,review))

print(">>> DOUBLE REVIEW TEST")

print_response(api.add_review(game_id,review))

print(">>> DELETE GAME REVIEW TEST")

review_id = get_user_game_review(user,game_id)

print_response(api.delete_review(review_id))

### Reviews Manipulation Corner Cases

In [ ]:
user = create_unique_user()
login_as(user)

impostor = general_user_2
game_id = get_game_id(general_game)

review = {
    "reviewText" : "i like apples, tangerines and "+ game,
    "score" : 9.5
}

print_response(api.add_review(game_id,review))

review_id = get_user_game_review(user,game_id,True)

api.logout()

print(">>> NO LOGIN REVIEWS MANIPULATION")

print_response(api.add_review(game_id,review))

print_response(api.delete_review(review_id))

print(">>> WRONG USER ATTEMPTING DELETE")
login_as(impostor)
print_response(api.delete_review(review_id))


# cleanup
login_as(user, False) 
api.delete_review(review_id)

## ADMIN CONTROLLER

### Game Manipulation

In [ ]:
admin_login()

fake = Faker()
name = fake.company() + f" {random.randint(1, 9)}" # avoid duplicates
print("Adding game: " + name)
payload = {
    "name": name,
    "releaseDate": "2002-08-29",
    "price": 100,
    "discount": 30,
    "description": "fortnite is bad",
    "imageURL": "obama.com",
    "supportedOS": [
        "Windows",
        "Linux"
    ],
    "achievements": 100,
    "developers": [
        "obamna"
    ],
    "publishers": [
        "epic games",
        "israel"
    ],
    "genres": [
        "super obama"
    ]
}

edit_payload = {
    "name" : name + " fortnite",
    "discount":35,
    "description":"fortnite is good",
    "imageURL":"obamadotcom",
    "supportedOS":["Windows","Linux","mac"],
    "achievements":300,
    "developers":["netanyahu"],
    "publishers":["israel"]
}

print(">>> ADD NEW GAME")
print_response(api.add_game(payload))

game_id = get_game_id(name)

print(">>> ADDING FAKE REVIEW")
user =  create_unique_user()
login_as(user)

review_payload = {
    "reviewText": "This game makes me want to eat cheese",
    "score":10
}
api.add_review(game_id,review_payload)



admin_login()
print(">>> REQUESTING GAME INFO")

print_response(api.get_game_info(game_id))


print(">>> ADDING EXISTING GAME")
print_response(api.add_game(payload))

print(">>> EDIT GAME")
print_response(api.edit_game(game_id, edit_payload))

print(">>> REQUESTING EDITED GAME INFO")

print_response(api.get_game_info(game_id))

print(">>> REQUESTING EDITED REVIEW")
print_response(api.get_user_reviews(get_user_id(user)))


print(">>> EDIT GAME TO SAME NAME")

print_response(api.edit_game(game_id, edit_payload))


print(">>> EDIT GAME TO EXISTING NAME")

edit_payload["name"] = general_game
print_response(api.edit_game(game_id, edit_payload))

print(">>> FRESH GAME DELETE")
print_response(api.delete_game(game_id))

print(">>> DELETE NON-EXISTING GAME")
print_response(api.delete_game(game_id))


### Fresh User Delete

In [ ]:
api.logout()

print(">>> DELETE USER AS ADMIN")

fake = Faker()

register_username = fake.user_name() # avoid duplicates
user_payload = {
    "username" : register_username,
    "email" : register_username + "@hotmail.com",
    "password" : register_username,
    "birthDate" : "2002-08-29"
}

print(">>>> NEW USER IS: " + register_username)
api.register(user_payload)

user_id = get_user_id(register_username)

admin_login()

print(">>> DELETE FRESH USER")
print_response(api.delete_user_admin(user_id))

print(">>> DELETE NON-EXISTENT USER")
print_response(api.delete_user_admin(user_id))


print(">>> REVIEW DELETION TEST")

# we first add a review to delete
game = general_game
user = create_unique_user()

print(">>>> NEW USER IS: " + user)
game_id = get_game_id(game)
login_as(user, False)

review = {
    "reviewText" : "i like apples, tangerines and "+ game,
    "score" : 9.5
}

api.add_review(game_id,review)
review_id = get_user_game_review(user,game_id,False)

admin_login()
print_response(api.delete_review(review_id))

print_response(api.delete_user_admin(get_user_id(user)))



### Authentication Test

In [ ]:
print(">>> ADMIN AUTH TEST")
login_as(general_user)

user_id = get_user_id(general_user_2)
game_id = get_game_id(general_game)

print_response(api.add_game({"payload" : "fake"}))
print_response(api.edit_game({"payload" : "fake"}))
print_response(api.delete_game(game_id))
print_response(api.delete_user_admin(user_id))

### Heavily Connected Game Deletion

In [ ]:
print(">>> HEAVILY CONNECTED GAME DELETION")

connected_users = 5

admin_login(False)

fake = Faker()
name = fake.company() + f"{random.randint(1000, 9999)}"
print("Adding game: " + name)
payload = {
    "name": name,
    "releaseDate": "2002-08-29",
    "price": 100,
    "discount": 30,
    "description": "fortnite is bad",
    "imageURL": "obama.com",
    "supportedOS": [
        "Windows",
        "Linux"
    ],
    "achievements": 100,
    "developers": [
        "obamna"
    ],
    "publishers": [
        "epic games",
        "israel"
    ],
    "genres": [
        "super obama"
    ]
}

review = {
    "reviewText" : "i like apples, tangerines and "+ name,
    "score" : 9.5
}

api.add_game(payload)
game_id = get_game_id(name)

library = {
    "gameId" : game_id,
    "achievements" : 100,
    "hoursPlayed" : 100
}

api.logout(False)

query_string = "{ username: { $in: [" 
neo4j_string = "OPTIONAL MATCH (anchor:Game {name : \"" + name + "\"}) \n"
relationship_string = "OPTIONAL MATCH p = (gamer:User)-[r:PLAYED]-(anchor) \nRETURN anchor, p"

user_list = []

for i in range(0,connected_users):
    
    u = create_unique_user()
    user_list.append(u)

    login_as(u, False)
    api.add_review(game_id,review)
    api.edit_library(library) 
    
    neo4j_string += f"MATCH (u{i}:User" + " {username : \"" + u + "\"}) \n"
    relationship_string += f", u{i}"


print()
print()
print_mongo_query(user_list)
print()

print("NEO4J QUERY:\n")
print(neo4j_string + relationship_string)
print()
print()

admin_login()

print_response(api.delete_game(game_id))

print(">>>> Cleanup...")

for u in user_list:
    api.delete_user_admin(get_user_id(u))

### Heavily Connected User Deletion

In [ ]:
api.logout(False)

friends = 5
print(f">>> USER DELETION WITH {friends} friends")

anchor = create_unique_user()
anchor_id = get_user_id(anchor)

print(">>>>> Anchor is: " + anchor)

queries = create_fake_friends(anchor,friends)

add_games_batch(anchor,[general_game, "ULTRAKILL"])


print()
print("MONGODB QUERY:\n")
print(queries["mongo"])
print()
print()

print("NEO4J QUERY:\n")
print(queries["neo4j"])
print()
print()

admin_login()

print(">>> DELETING ANCHOR")

print_response(api.delete_user_admin(anchor_id))

## INTERESTING QUERIES

### Game Recommendation

In [ ]:
print(">>> GAME RECOMMENDATION TEST")

friend_number = 10

anchor = create_unique_user()
anchor_id = get_user_id(anchor)
print(">>>>> ANCHOR = " + anchor)

create_fake_friends(anchor,friend_number)

friend_list = api.get_friend_list(anchor_id).json()["friends"]

temp = 0

for fr in friend_list:
    add_games_batch(fr["username"], ["Undertale"])
    if temp < 3:
        add_games_batch(fr["username"],["ULTRAKILL"])
    temp += 1
    
print("\nNEO4J QUERY:")
print("MATCH (u:User {username : \"" + anchor +"\"})-[r:FRIENDS_WITH]->(friend:User)-[p:PLAYED]->(recGame:Game) " +
           " WHERE NOT (u)-[:PLAYED]->(recGame) "+
          "  RETURN r, p, friend, recGame, u\n")

login_as(anchor)
print_response(api.get_game_recommendations())

print(">>>> Cleaning up...")

admin_login(False)
api.delete_user_admin(anchor_id)
for fr in friend_list:
    api.delete_user_admin(get_user_id(fr["username"]))



### Get Trending Games

In [ ]:
print(">>> TRENDING GAMES TEST")

api.logout()

print("\nNEO4J QUERY:")
print("MATCH (u1:User)-[f:FRIENDS_WITH]->(u2:User) \n" +
            " WHERE elementId(u1) < elementId(u2) \n" +
            " MATCH (u1)-[p1:PLAYED]->(g:Game)<-[p2:PLAYED]-(u2) \n" +
            " RETURN u1,u2,p1,p2,g,f\n")

values = [20, 11, 1, 30, 9]
games = ["ELDEN RING", "2Dark","God of War", "Sekiro™: Shadows Die Twice - GOTY Edition", "Alien Battlefield"]
random_user = []

for friend_group_type in ["net"]:#, "star"]:
    print("\n>>>> FRIEND GROUP TYPE: " + friend_group_type)
    for i in range(0,len(games)):
        random_user.append(create_unique_user())
        add_games_batch(random_user[i], [games[i]])
        create_fake_friends(random_user[i], values[i], 0)

        friend_list = api.get_friend_list(get_user_id(random_user[i])).json()["friends"]

        for fr in friend_list:
            add_games_batch(fr["username"], [games[i]])
            if(friend_group_type == "net"):    
                login_as(fr["username"], False)
                for f2 in friend_list:
                    if(f2["username"] != fr["username"]):
                        api.send_friend_request(get_user_id(f2["username"]))

                api.logout(False)

    admin_login(False)
    print_response(api.get_trending_games())

    print(">>> Cleaning up...\n")

    admin_login(False)

    #for i in range(0,len(random_user)):

    #    friend_list = api.get_friend_list(get_user_id(random_user[i])).json()["friends"]
        
    #    api.delete_user_admin(get_user_id(random_user[i]))

    #    for fr in friend_list:
    #        api.delete_user_admin(get_user_id(fr["username"]))

    

### Jaccard Similarity

## Miscellaneous

### Review Bombing

In [ ]:
api.logout()

print(">>> REVIEW BOMBING")
reviews_num = 5000
target = "ULTRAKILL"

target_id = get_game_id(target)

for i in range(0,reviews_num):
    user = create_unique_user()
    login_as(user,False)
    curr_review = reviews_num - i 
    payload = {"reviewText" : "this is the " + str(curr_review) +"th review!", "score" : 5 }
    api.add_review(target_id,payload)


    